# Modelling

## Load Packages

In [ ]:
from catboost import CatBoostClassifier
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torchmetrics.classification import BinaryCalibrationError
from xgboost import XGBClassifier
import joblib
import numpy as np
import os
import pandas as pd
import torch

## Set-up

In [2]:
seed = 123
os.makedirs("../models", exist_ok=True)

## Data Wrangling

In [3]:
features_dfs = []

for features_file in os.listdir("../data/features"):
    features_dfs.append(pd.read_parquet(f"../data/features/{features_file}"))

labels_dfs = []

for labels_file in os.listdir("../data/labels"):
    labels_dfs.append(pd.read_parquet(f"../data/labels/{labels_file}"))

X = pd.concat(features_dfs)
y = pd.concat(labels_dfs)

In [4]:
for col in X.select_dtypes(include="bool").columns:
    X[col] = X[col].astype("int8")

period_cols = [c for c in X.columns if "period" in c]

X[period_cols] = X[period_cols].astype("int8")

max_pitch_distance = np.sqrt(120**2 + 80**2)

for col in [
    "actor_minimum_distance_to_opponent",
    "actor_minimum_distance_to_teammate",
    "actor_mean_distance_to_opponent",
    "actor_mean_distance_to_teammate",
]:
    X[col] = X[col].fillna(max_pitch_distance)

for col in [
    "actor_stdev_distance_to_opponent",
    "actor_stdev_distance_to_teammate",
    "opponent_convex_hull_area",
    "teammate_convex_hull_area"
]:
    X[col] = X[col].fillna(0)

In [5]:
id_cols = ["game_id", "action_id"]

manual_spatial_feature_cols = [
    'actor_minimum_distance_to_opponent',
    'actor_minimum_distance_to_teammate',
    'actor_mean_distance_to_opponent',
    'actor_mean_distance_to_teammate',
    'actor_stdev_distance_to_opponent',
    'actor_stdev_distance_to_teammate',
    'number_of_opponents_within_radius_of_actor',
    'number_of_teammates_within_radius_of_actor',
    'opponent_convex_hull_area',
    'teammate_convex_hull_area',
]

baseline_feature_cols = [
    c for c in X.columns
    if c not in id_cols + manual_spatial_feature_cols and not c.startswith("spatial_embedding_")
]

spatial_embedding_feature_cols = [
    c for c in X.columns
    if c.startswith("spatial_embedding_")
]

X_model_baseline = X[baseline_feature_cols].copy()
X_model_manual_spatial_features = X[baseline_feature_cols + manual_spatial_feature_cols].copy()
X_model_embeddings = X[baseline_feature_cols + spatial_embedding_feature_cols].copy()
X_model_all = X[baseline_feature_cols + manual_spatial_feature_cols + spatial_embedding_feature_cols].copy()

y_scores = y["scores"].astype(int)
y_concedes = y["concedes"].astype(int)

In [6]:
game_ids = X["game_id"].unique()

train_games, test_games = train_test_split(
    game_ids,
    test_size=0.2,
    random_state=seed,
)

train_mask = X["game_id"].isin(train_games)
test_mask = X["game_id"].isin(test_games)

X_train_baseline = X_model_baseline.loc[train_mask]
X_train_manual_spatial_features = X_model_manual_spatial_features.loc[train_mask]
X_train_embeddings = X_model_embeddings.loc[train_mask]
X_train_all = X_model_all.loc[train_mask]

X_test_baseline = X_model_baseline.loc[test_mask]
X_test_manual_spatial_features = X_model_manual_spatial_features.loc[test_mask]
X_test_embeddings = X_model_embeddings.loc[test_mask]
X_test_all = X_model_all.loc[test_mask]

y_scores_train = y_scores.loc[train_mask]
y_scores_test = y_scores.loc[test_mask]

y_concedes_train = y_concedes.loc[train_mask]
y_concedes_test = y_concedes.loc[test_mask]

In [7]:
feature_sets = {
    "baseline": X_model_baseline.columns,
    "manual": X_model_manual_spatial_features.columns,
    "embeddings": X_model_embeddings.columns,
    "all": X_model_all.columns,
}

X_train = {
    name: X_model_all.loc[train_mask, cols].copy()
    for name, cols in feature_sets.items()
}

X_test = {
    name: X_model_all.loc[test_mask, cols].copy()
    for name, cols in feature_sets.items()
}

targets_train = {
    "scores": y_scores.loc[train_mask],
    "concedes": y_concedes.loc[train_mask],
}

targets_test = {
    "scores": y_scores.loc[test_mask],
    "concedes": y_concedes.loc[test_mask],
}

In [16]:
joblib.dump(X_train, "../data/modelling/train/X_train_dict.pkl")
joblib.dump(X_test, "../data/modelling/test/X_test_dict.pkl")
joblib.dump(targets_train, "../data/modelling/train/targets_train_dict.pkl")
joblib.dump(targets_test, "../data/modelling/test/targets_test_dict.pkl")

['../data/modelling/test/targets_test_dict.pkl']

## Modelling

In [8]:
model_defs = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            solver="lbfgs",
            max_iter=1000,
            random_state=seed
        ))
    ]),

    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=3,
        random_state=seed
    ),

    "xgboost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=seed
    ),

    "catboost": CatBoostClassifier(
        random_seed=seed,
        verbose=False
    ),
}

In [9]:
models = {}
model_dir = "../models/classifiers"
os.makedirs(model_dir, exist_ok=True)

for model_name, model_template in model_defs.items():

    for feature_name, cols in feature_sets.items():

        X_train_set = X_train[feature_name]
        X_test_set = X_test[feature_name]

        for target_name, y_train_set in {
            "scores": y_scores_train,
            "concedes": y_concedes_train
        }.items():

            print(f"Training {model_name} | {feature_name} | {target_name}")

            model = clone(model_template)
            model.fit(X_train_set, y_train_set)

            models[(model_name, feature_name, target_name)] = model

            joblib.dump(
                model,
                f"{model_dir}/{model_name}_{feature_name}_{target_name}.pkl"
            )

Training logistic_regression | baseline | scores
Training logistic_regression | baseline | concedes
Training logistic_regression | manual | scores
Training logistic_regression | manual | concedes
Training logistic_regression | embeddings | scores
Training logistic_regression | embeddings | concedes
Training logistic_regression | all | scores
Training logistic_regression | all | concedes
Training random_forest | baseline | scores
Training random_forest | baseline | concedes
Training random_forest | manual | scores
Training random_forest | manual | concedes
Training random_forest | embeddings | scores
Training random_forest | embeddings | concedes
Training random_forest | all | scores
Training random_forest | all | concedes
Training xgboost | baseline | scores
Training xgboost | baseline | concedes
Training xgboost | manual | scores
Training xgboost | manual | concedes
Training xgboost | embeddings | scores
Training xgboost | embeddings | concedes
Training xgboost | all | scores
Training

In [10]:
results = []
bce = BinaryCalibrationError()

for model_name, model_template in model_defs.items():

    for feature_name, cols in feature_sets.items():

        X_test_set = X_test[feature_name]

        for target_name, y_test_set in {
            "scores": y_scores_test,
            "concedes": y_concedes_test
        }.items():

            model = models[(model_name, feature_name, target_name)]

            probs = model.predict_proba(X_test_set)[:, 1]

            results.append({
                "model": model_name,
                "feature_set": feature_name,
                "target": target_name,
                "brier_score": brier_score_loss(y_test_set, probs),
                "log_loss": log_loss(y_test_set, probs),
                "ece": bce(
                    torch.tensor(probs, dtype=torch.float32),
                    torch.tensor(y_test_set.values, dtype=torch.int64)
                ).item(),
                "auc": roc_auc_score(y_test_set, probs),
            })

In [11]:
results_df = (
    pd.DataFrame(results)
    .sort_values(
        ["model", "target", "feature_set"]
    )
)

results_df.to_parquet("../results/model_training_results.parquet", compression="zstd", index=False)